<a href="https://colab.research.google.com/github/Zhalil24/BreastMRI-CNN-V-T-Classification/blob/main/BaseLineVITModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import gc
import os
import random
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# Exactly the same controlled protocol as cnn_baseline_equal_protocol.py
DATA_DIR = "/content/breast_mri_dataset"
RAR_SOURCE = "/content/drive/MyDrive/data_set_zip/breast_mri_dataset.rar"
OUTPUT_ROOT = Path("/content/drive/MyDrive/vision_transformers_baseline_equal_protocol_outputs")
IMG_SIZE, BATCH_SIZE, NUM_EPOCHS, NUM_CLASSES = 224, 32, 10, 2
LEARNING_RATE, N_SPLITS, RANDOM_STATE = 1e-3, 5, 42
MODEL_NAMES = ["ViT-B-16", "ViT-B-32", "ViT-L-16", "ViT-L-32", "Swin-T"]


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


set_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
drive.mount("/content/drive")
local_rar = "/content/breast_mri_dataset.rar"
shutil.copy2(RAR_SOURCE, local_rar)
subprocess.run(["unrar", "x", "-o+", local_rar, "/content/"], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=transform)
test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=transform)
assert train_dataset.class_to_idx == test_dataset.class_to_idx, "Train/test class mapping differs."
assert len(train_dataset.classes) == NUM_CLASSES, "This script expects two classes."
print("Class mapping:", train_dataset.class_to_idx)
if train_dataset.class_to_idx.get("Malignant") != 1:
    print("WARNING: Class 1 is treated as positive. Confirm that class 1 is Malignant.")
class_names = train_dataset.classes
targets = np.asarray(train_dataset.targets)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=torch.cuda.is_available())
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(np.zeros(len(targets)), targets))
fold_manifest = []
for fold, (tr_idx, va_idx) in enumerate(FOLDS, 1):
    for split, indices in (("Train", tr_idx), ("Validation", va_idx)):
        for idx in indices:
            fold_manifest.append({"Fold": fold, "Split": split,
                                  "File": train_dataset.samples[idx][0], "Label": int(targets[idx])})
pd.DataFrame(fold_manifest).to_csv(OUTPUT_ROOT / "Fold Assignments.csv", index=False)


def build_model(name):
    builders = {
        "ViT-B-16": (models.vit_b_16, models.ViT_B_16_Weights.IMAGENET1K_V1),
        "ViT-B-32": (models.vit_b_32, models.ViT_B_32_Weights.IMAGENET1K_V1),
        "ViT-L-16": (models.vit_l_16, models.ViT_L_16_Weights.IMAGENET1K_V1),
        "ViT-L-32": (models.vit_l_32, models.ViT_L_32_Weights.IMAGENET1K_V1),
        "Swin-T": (models.swin_t, models.Swin_T_Weights.IMAGENET1K_V1),
    }
    builder, weights = builders[name]
    model = builder(weights=weights)
    for p in model.parameters(): p.requires_grad = False
    if name == "Swin-T":
        model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    else:
        model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
    return model.to(device)


def train_epoch(model, loader, optimizer, criterion):
    # eval() disables dropout/stochastic depth in the frozen backbone while
    # gradients continue to update the linear classification head.
    model.eval(); total_loss = correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True); outputs = model(images); loss = criterion(outputs, labels)
        loss.backward(); optimizer.step(); total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item(); total += labels.size(0)
    return total_loss / total, correct / total


def evaluate_loss_accuracy(model, loader, criterion):
    model.eval(); total_loss = correct = total = 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device); outputs = model(images)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item(); total += labels.size(0)
    return total_loss / total, correct / total


def predict(model, loader):
    model.eval(); ys, ps, probs = [], [], []
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images.to(device)); probability = outputs.softmax(1)[:, 1]
            ys.extend(labels.numpy()); ps.extend(outputs.argmax(1).cpu().numpy()); probs.extend(probability.cpu().numpy())
    return np.asarray(ys), np.asarray(ps), np.asarray(probs)


def metrics(y, pred, prob):
    try: auc = roc_auc_score(y, prob)
    except ValueError: auc = np.nan
    return {"Accuracy": accuracy_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0), "F1-Score": f1_score(y, pred, zero_division=0),
            "ROC-AUC": auc}


def save_learning_curves(history, name, folder):
    epochs = range(1, len(history["Train Accuracy"]) + 1)
    for value in ("Accuracy", "Loss"):
        plt.figure(figsize=(8, 5))
        plt.plot(epochs, history[f"Train {value}"], marker="o", label=f"Train {value}")
        plt.plot(epochs, history[f"Validation {value}"], marker="o", label=f"Validation {value}")
        plt.title(f"{name} - {value}"); plt.xlabel("Epoch"); plt.ylabel(value)
        if value == "Accuracy": plt.ylim(0, 1)
        plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
        plt.savefig(folder / f"{name} - {value}.png", dpi=300); plt.close()


def save_split_plots(y, pred, prob, result, name, split, folder):
    cm = confusion_matrix(y, pred, labels=[0, 1]); plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{name} - {split} Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout(); plt.savefig(folder / f"{name} - {split} Confusion Matrix.png", dpi=300); plt.close()
    if not np.isnan(result["ROC-AUC"]):
        fpr, tpr, _ = roc_curve(y, prob); plt.figure(figsize=(7, 5))
        plt.plot(fpr, tpr, label=f"AUC = {result['ROC-AUC']:.4f}"); plt.plot([0, 1], [0, 1], "--")
        plt.title(f"{name} - {split} ROC Curve"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
        plt.savefig(folder / f"{name} - {split} ROC Curve.png", dpi=300); plt.close()
    plt.figure(figsize=(8, 5)); bars = plt.bar(result.keys(), result.values(), color="#1f77b4"); plt.ylim(0, 1)
    plt.title(f"{name} - {split} Metrics"); plt.ylabel("Score")
    for bar, value in zip(bars, result.values()):
        plt.text(bar.get_x() + bar.get_width()/2, min(value + .015, .98), f"{value:.4f}", ha="center")
    plt.tight_layout(); plt.savefig(folder / f"{name} - {split} Metrics.png", dpi=300); plt.close()


def run_model(name):
    model_dir = OUTPUT_ROOT / name; model_dir.mkdir(exist_ok=True)
    history_rows, split_rows = [], {"Train": [], "Validation": [], "Test": []}
    criterion = nn.CrossEntropyLoss()
    for fold, (train_idx, val_idx) in enumerate(FOLDS, 1):
        set_seed(RANDOM_STATE + fold)
        fold_dir = model_dir / f"Fold {fold}"; fold_dir.mkdir(exist_ok=True)
        train_generator = torch.Generator().manual_seed(RANDOM_STATE + fold)
        train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=2, pin_memory=torch.cuda.is_available(), generator=train_generator)
        train_eval_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=False,
                                       num_workers=2, pin_memory=torch.cuda.is_available())
        val_loader = DataLoader(Subset(train_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=2, pin_memory=torch.cuda.is_available())
        model = build_model(name)
        optimizer = optim.Adam((p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE)
        history = {"Train Loss": [], "Train Accuracy": [], "Validation Loss": [], "Validation Accuracy": []}
        best_acc, best_epoch, best_state = -1.0, 1, None
        for epoch in range(1, NUM_EPOCHS + 1):
            tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
            va_loss, va_acc = evaluate_loss_accuracy(model, val_loader, criterion)
            for key, value in (("Train Loss", tr_loss), ("Train Accuracy", tr_acc),
                               ("Validation Loss", va_loss), ("Validation Accuracy", va_acc)):
                history[key].append(value)
            history_rows.append({"Model": name, "Fold": fold, "Epoch": epoch, "Train Loss": tr_loss,
                                 "Train Accuracy": tr_acc, "Validation Loss": va_loss, "Validation Accuracy": va_acc})
            if va_acc > best_acc:
                best_acc, best_epoch = va_acc, epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"{name} | Fold {fold} | Epoch {epoch}/{NUM_EPOCHS} | Train Acc {tr_acc:.4f} | Val Acc {va_acc:.4f}")
        model.load_state_dict(best_state); save_learning_curves(history, name, fold_dir)
        for split, loader in (("Train", train_eval_loader), ("Validation", val_loader), ("Test", test_loader)):
            y, pred, prob = predict(model, loader); result = metrics(y, pred, prob)
            split_rows[split].append({"Model": name, "Fold": fold, "Best Epoch": best_epoch, **result})
            save_split_plots(y, pred, prob, result, name, split, fold_dir)
        torch.save(best_state, fold_dir / f"{name} - Model.pth")
        del model, optimizer, best_state; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    summary = []
    for split, rows in split_rows.items():
        frame = pd.DataFrame(rows)
        for metric_name in ("Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"):
            summary.append({"Model": name, "Split": split, "Metric": metric_name,
                            "Mean": frame[metric_name].mean(), "Standard Deviation": frame[metric_name].std()})
    with pd.ExcelWriter(model_dir / f"{name} - Results.xlsx", engine="openpyxl") as writer:
        pd.DataFrame(history_rows).to_excel(writer, sheet_name="Epoch History", index=False)
        for split, rows in split_rows.items(): pd.DataFrame(rows).to_excel(writer, sheet_name=f"{split} Results", index=False)
        pd.DataFrame(summary).to_excel(writer, sheet_name="Mean and Std", index=False)
    return summary


all_summary = []
for model_name in MODEL_NAMES:
    all_summary.extend(run_model(model_name))
pd.DataFrame(all_summary).to_excel(OUTPUT_ROOT / "Transformers - Combined Results.xlsx", index=False)
print("All transformer baseline experiments completed:", OUTPUT_ROOT)
